# Data Loading, Storage, and File Formats 

Reading data and making it accessible (often called *data loading*) is a necessary first step for using most of the tools. The term **parsing** is also sometimes used to describe loading text data and interpreting it as tables and different data types. 

Going to focus more on data input and output using pandas, though there are numerous tools in other libraries to help with reading and writing data in various formats. 

Input and output typically fall into a few main categories:
- reading text files and other more efficient on-disk formats 
- loading data from databases
- interacting with network sources like web APIs


## Reading and Writing Data in Text Format 

pandas features a number of functions for reading tabular data as DataFrame object. 

``pandas.read_csv`` is one of the most frequently used. 

| Function | Description |
|---|---|
| `read_csv` | Load delimited data from a file, URL, or file-like object; use comma as default delimiter |
| `read_fwf` | Read data in fixed-width column format (i.e., no delimiters) |
| `read_clipboard` | Variation of `read_csv` that reads data from the clipboard; useful for converting tables from web pages |
| `read_excel` | Read tabular data from an Excel XLS or XLSX file |
| `read_hdf` | Read HDF5 files written by pandas |
| `read_html` | Read all tables found in the given HTML document |
| `read_json` | Read data from a JSON (JavaScript Object Notation) string representation |
| `read_feather` | Read the Feather binary file format |
| `read_orc` | Read the Apache ORC binary file format |
| `read_parquet` | Read the Apache Parquet binary file format |
| `read_pickle` | Read an object stored by pandas using the Python pickle format |
| `read_sas` | Read a SAS dataset stored in one of the SAS system's custom storage formats |
| `read_spss` | Read a data file created by SPSS |
| `read_sql` | Read the results of a SQL query (using SQLAlchemy) |
| `read_sql_table` | Read an entire SQL table (using SQLAlchemy); equivalent to using a query that selects everything in that table using `read_sql` |
| `read_stata` | Read a dataset from Stata file format |
| `read_xml` | Read a table of data from an XML file |

**Indexing**
- Can treat one or more columns as the returned DataFrame, and whether to get column names from the file, arguments you provide, or not at all . 

**Type inference and data conversion**
- Includes the user-defined value conversions and custom list of missing value markers.

**Data and time parsing** 
- Includes a combining capability, including combining data and time information spread over multiple columns into a single column in the result.

**Inserting**
- Support for iterating over chunks of very large files. 

**Unclean data issues**
- Includes skipping rows or a footer, comments, or the other minor things like numeric data with thousands separated by commas


Because of how messy data is in the real world, some of the data loading functions (like ``pandas.read_csv``) have accumulated a long list of optional arguments over time. 

Some of these functions perform **type inference**, because the column data types are not part of the data format. That means you don't necessarily have to specify which columns are numeric, integer, Boolean, or string. Other data formats, like HDF5, ORC, and Parquet, have the data type information embedded in the format.


Handling dates and other custom types can require extra effort. 

We can start with Comma-Seperated Values (CSV) text file:

In [70]:
import pandas as pd 

In [71]:
df = pd.read_csv("/Users/bmart231/DS/examples/ex1.csv")

df

,a,b,c,d,message
0,1,2,3,4,hello
1,5,6,7,8,world
2,9,10,11,12,foo


A file will not always have a header row. Consider this file:

You can specify names yourself:

In [72]:
pd.read_csv("/Users/bmart231/DS/examples/ex2.csv", header= None)

,0,1,2,3,4
0,1,2,3,4,hello
1,5,6,7,8,world
2,9,10,11,12,foo


In [73]:
pd.read_csv("/Users/bmart231/DS/examples/ex2.csv", names=["a", "b", "c", "d", "message"])

,a,b,c,d,message
0,1,2,3,4,hello
1,5,6,7,8,world
2,9,10,11,12,foo


Suppose you wanted the ``message`` column to be the index of the returned DataFrame. You can either indicate you want the column at index 4 or named ``"message"`` using the ``index_col`` argument: 

In [74]:
names = ["a", "b", "c", "d", "message"]

pd.read_csv("/Users/bmart231/DS/examples/ex2.csv", names = names, index_col="message")

,a,b,c,d
message,,,,
hello,1,2,3,4
world,5,6,7,8
foo,9,10,11,12


If you want to form a hierarchial index from mulitple columns, pass a list of columns numbers or names:

In [75]:
parsed = pd.read_csv("/Users/bmart231/DS/examples/csv_mindex.csv", index_col=["key1", "key2"])

parsed

value1  value2
key1 key2                
one  a          1       2
     b          3       4
     c          5       6
     d          7       8
two  a          9      10
     b         11      12
     c         13      14
     d         15      16

In some cases, a table might not have a fixed dilimiter, using whitespace or some other pattern to seperate fields. Consider a text file that looks like this:

While you could do some munging by hand, the fields here are separated by a variable amount of whitespace. In these cases, you can pass a regular expression as a delimiter for ``pandas.read_csv``. This can be expressed by the regular expression /s+, so we have then:

In [76]:
result = pd.read_csv("/Users/bmart231/DS/examples/ex3.txt", sep=r"\s+")

result

,A,B,C
aaa,-0.264438,-1.026059,-0.619500
bbb,0.927272,0.302904,-0.032399
ccc,-0.264273,-0.386314,-0.217601
ddd,-0.871858,-0.348382,1.100491


Because there was one fewer column name than the number of data rows, ``pandas.read_csv`` infer that the first column should be the DataFrame's index in this special case. 

The file parsing functions have many additional arguments to help you handle the wide variety of exception file formats that occur. For example, you can skip the first, third, and fourth rows of a file with ``skiprows``:

In [77]:
pd.read_csv("/Users/bmart231/DS/examples/ex4.csv", skiprows=[0, 2, 3])

,a,b,c,d,message
0,1,2,3,4,hello
1,5,6,7,8,world
2,9,10,11,12,foo


Handling missing values is an important and frequently nuanced part of the file reading process. Missing data is usually either not present (empty string) or marked by some **sentinel** (placeholder) value. By default, pandas uses a set of commonly occuring sentinels, such as ``NA`` and ``Null``:

In [78]:
result = pd.read_csv("/Users/bmart231/DS/examples/ex5.csv")

result

,something,a,b,c,d,message
0,one,1,2,3.0,4,NaN
1,two,5,6,NaN,8,world
2,three,9,10,11.0,12,foo


Recall that pandas outputs missing values as ``NaN``, so we have two null or missing values in ``result``:

In [79]:
pd.isna(result)

,something,a,b,c,d,message
0,False,False,False,False,False,True
1,False,False,False,True,False,False
2,False,False,False,False,False,False


The ``na_values`` option accepts a sequence of strings to add the default list of strings recognized as missing:

In [80]:
result = pd.read_csv("/Users/bmart231/DS/examples/ex5.csv", na_values=["NULL"])

result

,something,a,b,c,d,message
0,one,1,2,3.0,4,NaN
1,two,5,6,NaN,8,world
2,three,9,10,11.0,12,foo


In [81]:
result2 = pd.read_csv("/Users/bmart231/DS/examples/ex5.csv", keep_default_na=False)

result2

,something,a,b,c,d,message
0,one,1,2,3,4,NA
1,two,5,6,,8,world
2,three,9,10,11,12,foo


In [82]:
result2.isna()

,something,a,b,c,d,message
0,False,False,False,False,False,False
1,False,False,False,False,False,False
2,False,False,False,False,False,False


In [83]:
result3 = pd.read_csv("/Users/bmart231/DS/examples/ex5.csv", keep_default_na=False, na_values=["NA"])

result3

,something,a,b,c,d,message
0,one,1,2,3,4,NaN
1,two,5,6,,8,world
2,three,9,10,11,12,foo


In [84]:
result3.isna()

,something,a,b,c,d,message
0,False,False,False,False,False,True
1,False,False,False,False,False,False
2,False,False,False,False,False,False


Different NA sentinels can be specified for each column in a dictionary:

In [85]:
sentinels = {"message": ["foo", "message"], "something": ["two"]}

pd.read_csv("/Users/bmart231/DS/examples/ex5.csv", na_values=sentinels, keep_default_na=False)

,something,a,b,c,d,message
0,one,1,2,3,4,NA
1,NaN,5,6,,8,world
2,three,9,10,11,12,NaN


Frequently used options in ``pandas.read_csv``.

| Argument | Description |
|---|---|
| `path` | String indicating filesystem location, URL, or file-like object |
| `sep` or `delimiter` | Character sequence or regular expression to use to split fields in each row |
| `header` | Row number to use as column names; defaults to 0 (first row), but should be `None` if there is no header row |
| `index_col` | Column numbers or names to use as the row index in the result; can be a single name/number or a list for a hierarchical index |
| `names` | List of column names for result, combine with `header=None` |
| `skiprows` | Number of rows at beginning of file to ignore, or a list of row numbers (starting from 0) to skip |
| `na_values` | Sequence of values to replace with NA. They are added to the default list unless `keep_default_na=False` is passed |
| `keep_default_na` | Whether to use the default NA value list or not (`True` by default) |
| `comment` | Character(s) to split comments off the end of lines |
| `parse_dates` | Attempt to parse data to `datetime`; `False` by default. If `True`, will attempt to parse all columns. Otherwise can specify a list of column numbers or names to parse. If a list element is a tuple or list, will combine multiple columns together and parse to date (e.g., if date/time split across two columns) |
| `keep_date_col` | If joining columns to parse date, keep the joined columns; `False` by default |
| `converters` | Dictionary containing column number or name mapping to functions (e.g., `{"foo": f}` would apply the function `f` to all values in the `foo` column) |
| `dayfirst` | When parsing potentially ambiguous dates, treat as international format (e.g., 7/6/2012 → June 7, 2012); `False` by default |
| `date_parser` | Function to use to parse dates |
| `nrows` | Number of rows to read from beginning of file (not counting the header) |
| `iterator` | Return a `TextFileReader` object for reading the file piecemeal. This object can also be used with the `with` statement |
| `chunksize` | For iteration, size of file chunks |
| `skip_footer` | Number of lines to ignore at end of file |
| `verbose` | Print various parsing information, like the number of missing values placed in non-numeric columns |
| `encoding` | Text encoding (e.g., `"utf-8"` for UTF-8 encoded text). Defaults to `"utf-8"` if `None` |
| `squeeze` | If the parsed data contains only one column, return a Series |
| `thousands` | Separator for thousands (e.g., `","` or `"."`) |
| `decimal` | Decimal separator in numbers (e.g., `"."` or `","`) |
| `engine` | CSV parsing and conversion engine to use; can be one of `"c"`, `"python"`, or `"pyarrow"`. The C and pyarrow engines are faster, while the Python engine is currently more feature-complete |

## Reading Text Files in Pieces 

When processing very large files or figuring out the right set of arguments to correctly process a large file, you may want to read only a small piece of a file or iterate through smaller chunks of the file. 

Before we look at a larger file, we make the pandas display settings more compact:

In [86]:
pd.options.display.max_rows = 10

In [87]:
result = pd.read_csv("/Users/bmart231/DS/examples/ex6.csv")

result

,one,two,three,four,key
0,0.467976,-0.038649,-0.295344,-1.824726,L
1,-0.358893,1.404453,0.704965,-0.200638,B
2,-0.501840,0.659254,-0.421691,-0.057688,G
3,0.204886,1.074134,1.388361,-0.982404,R
4,0.354628,-0.133116,0.283763,-0.837063,Q
...,...,...,...,...,...
9995,2.311896,-0.417070,-1.409599,-0.515821,L
9996,-0.479893,-0.650419,0.745152,-0.646038,E
9997,0.523331,0.787112,0.486066,1.093156,K
9998,-0.362559,0.598894,-1.843201,0.887292,G


The elipsis marks ... indicates that rows in the middle of the DataFrame have been omitted. 

If you want to read only a small number of rows (avoiding reading the entire file), specify that with ``nrows``:

In [88]:
pd.read_csv("/Users/bmart231/DS/examples/ex6.csv", nrows=5)

,one,two,three,four,key
0,0.467976,-0.038649,-0.295344,-1.824726,L
1,-0.358893,1.404453,0.704965,-0.200638,B
2,-0.501840,0.659254,-0.421691,-0.057688,G
3,0.204886,1.074134,1.388361,-0.982404,R
4,0.354628,-0.133116,0.283763,-0.837063,Q


To read file in pieces, specify a ``chunksize`` as a number of rows:

In [89]:
chunker = pd.read_csv("/Users/bmart231/DS/examples/ex6.csv", chunksize=1000)

type(chunker)

pandas.io.parsers.readers.TextFileReader

The ``TextFileReader`` object returned by ``pandas.read_csv`` allows you to iterate over the parts of the file according to the ``chunksize`` . For example, we can iterate over ``ex6.csv``, aggregating the value counts in the "``key``" column, like so:

In [90]:
chunker = pd.read_csv("/Users/bmart231/DS/examples/ex6.csv", chunksize=1000)

tot = pd.Series([], dtype='int64')
for piece in chunker:
    tot = tot.add(piece["key"].value_counts(), fill_value=0)
    
tot = tot.sort_values(ascending=False)

In [91]:
tot[:10]

key
E    368.0
X    364.0
L    346.0
O    343.0
Q    340.0
M    338.0
J    337.0
F    335.0
K    334.0
H    330.0
dtype: float64

Data can also be exported to a delimiter format. Let's consider one of the CSV files read before:

In [92]:
data = pd.read_csv("/Users/bmart231/DS/examples/ex5.csv")

data

,something,a,b,c,d,message
0,one,1,2,3.0,4,NaN
1,two,5,6,NaN,8,world
2,three,9,10,11.0,12,foo


Using DataFrame's ``to_csv`` method, we can write the data out to a comma-seperated file:

In [93]:
data.to_csv("/Users/bmart231/DS/examples/out.csv")

Other delimiters can be used, of course (writing to ``sys.stdout`` so it prints the text result to the console rather than a file):

In [94]:
import sys 

data.to_csv(sys.stdout, sep ="|")

|something|a|b|c|d|message
0|one|1|2|3.0|4|
1|two|5|6||8|world
2|three|9|10|11.0|12|foo


Missing values appear as empty strings in the output. You might want to denote them by some other sentinel value:

In [95]:
import sys
data.to_csv(sys.stdout, na_rep="NULL")

,something,a,b,c,d,message
0,one,1,2,3.0,4,NULL
1,two,5,6,NULL,8,world
2,three,9,10,11.0,12,foo


While no other options specified, both the row and column labels are written. Both of these can be disabled:

In [96]:
data.to_csv(sys.stdout, index = False, header=False)

one,1,2,3.0,4,
two,5,6,,8,world
three,9,10,11.0,12,foo


You can also write only a subset of the columns, and in an order of your choosing:

In [97]:
data.to_csv(sys.stdout, index = False, columns=["a", "b", "c"])

a,b,c
1,2,3.0
5,6,
9,10,11.0


## Working with Other Delimited Formats

It's possible to load most forms of tabular data from disk using functions like ``pandas.read_csv``. In some cases, however, some manual processing may be necessary. It's not uncommon to recieve a file with one or more malformed lines that trip up ``pandas.read_csv``. To illustrate the basic tools, consider a small CSV file:

For any file with a single-character delimter, you can use Python's built-in csv module. To use it, pass any open file or file-like object to ``csv.reader``. 

In [98]:
import csv 

f = open("/Users/bmart231/DS/examples/ex7.csv")

reader = csv.reader(f)

Iterating through the reader like a file yields lists of values with any quote characters removed:

In [99]:
for line in reader:
    print(line)
    


['a', 'b', 'c']
['1', '2', '3']
['1', '2', '3']


From there, it's up to you to do the wrangling necessary to put the data in the form that you need. 


First, we read the file into a list of lines:

In [100]:
with open("/Users/bmart231/DS/examples/ex7.csv") as f:
    lines = list(csv.reader(f))

Then we split the lines into the header line and the data lines:

In [101]:
header, values = lines[0], lines[1:]

Then we can create a dictionary of data columns using a dictionary comprehension and the expression ``zip(*values)`` (beware that this will use a lot of memory on large files), which transposes rows to columns:

In [102]:
data_dict = {h: v for h, v in zip(header, zip(*values))}

data_dict

{'a': ('1', '1'), 'b': ('2', '2'), 'c': ('3', '3')}

CSV files come in many different flavors. To define a new format with a different delimiter, string quoting convention, or line terminator, we could define a simple subclass of ``csv.Dialect``:


In [105]:
class my_dialect(csv.Dialect):
    lineterminator = "\n"
    delimiter = ":"
    quotechar = ' " '
    quoting = csv.QUOTE_MINIMAL
    
reader = csv.reader(f, dialect = my_dialect)

ValueError: I/O operation on closed file.

We could also give a individual CSV dialect parameters as keywords to ``csv.reader`` without having to define a subclass:

In [104]:
reader = csv.reader(f, delimiter="|")

ValueError: I/O operation on closed file.

## JSON Data 

JSON (short for JavaScript Object Notation) has become one of the standard formats for sending data by HTTP request between web browsers and other applications. It is a much free-form data format than a tabular text form like CSV. Here is an example:

In [107]:
obj = """
{"name": "Wes",
"cities_lived": ["Akron", "Nashville", "New York", "San Francisco"],
"pet": null,
"siblings": [{"name": "Scott", "age": 34, "hobbies": ["guitars", "soccer"]},
{"name": "Katie", "age": 42, "hobbies": ["diving", "art"]}]
}
"""

JSON is very nearly valid Python code with the exception of its null value ``null`` and some other nuances. The basic types are obects (dictionaries), arrays (lists), strings, numbers, Booleans, and nulls. There are several Python libraries for reading and writing JSON data. To convert a JSON string to Python form, use ``json.loads``:

In [108]:
import json
result = json.loads(obj)

result 

{'name': 'Wes',
 'cities_lived': ['Akron', 'Nashville', 'New York', 'San Francisco'],
 'pet': None,
 'siblings': [{'name': 'Scott', 'age': 34, 'hobbies': ['guitars', 'soccer']},
  {'name': 'Katie', 'age': 42, 'hobbies': ['diving', 'art']}]}

``json.dumps`` on the other hand converts a Python object back to JSON:

In [109]:
asjson = json.dumps(result)

asjson



'{"name": "Wes", "cities_lived": ["Akron", "Nashville", "New York", "San Francisco"], "pet": null, "siblings": [{"name": "Scott", "age": 34, "hobbies": ["guitars", "soccer"]}, {"name": "Katie", "age": 42, "hobbies": ["diving", "art"]}]}'

How you convert to JSON object or list of objects to a DataFrame or some other data structure for analysis will be up to you. 

In [110]:
siblings = pd.DataFrame(result["siblings"], columns = ["names", "age"])

siblings

,names,age
0,NaN,34
1,NaN,42
